# Phase 4: PPO Training with Causal Diversity (MIND-large)

Trains a Proximal Policy Optimization (PPO) agent that ranks candidate news items by balancing click-through rate (CTR) and causal diversity impact (CDI).

## Tuning for MIND-large
- MIND-large provides more training sessions, leading to better policy generalization.
- Training uses a subset of sessions for the initial pass. Increase `total_timesteps` for full convergence.
- The same environment configuration (w=0.6, K=20, T=1) is used as in MIND-small for consistency.

## Inputs
- `data/scm_train.parquet` â€” Phase 1 feature table (MIND-large)
- `artifacts/cdi_cache.pkl` â€” Phase 3 precomputed causal diversity scores

## Outputs
- `artifacts/checkpoints/ppo_causal_rs_large_w03.zip` â€” trained PPO policy
- `artifacts/tb_logs/` â€” TensorBoard logs

In [ ]:
import warnings
import logging
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

_cwd = Path.cwd()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.rl_agent.train_ppo import train_ppo

warnings.filterwarnings('ignore')
logging.getLogger('src.rl_agent').setLevel(logging.INFO)

DATA = _root / "data"
ARTIFACTS = _root / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(f"Project root: {_root}")
print(f"Dataset: MIND-large")
print(f"Artifacts: {ARTIFACTS}")

## 2. Load Data and CDI Cache

In [ ]:
import pyarrow.parquet as pq

data_path = DATA / "scm_train.parquet"
if data_path.is_dir():
    parts = sorted(data_path.glob("*.parquet"))
    dfs = []
    for p in parts:
        tbl = pq.read_table(p)
        dfs.append(tbl.to_pandas())
        del tbl
    df = pd.concat(dfs, ignore_index=True)
    del dfs
    import gc; gc.collect()
elif data_path.is_file():
    df = pd.read_parquet(data_path)
else:
    raise FileNotFoundError(f"{data_path} not found")
print(f"Loaded MIND-large training data: {len(df)} rows, {df['user_id'].nunique()} users")

# Try full CDI cache first, then fall back to subset cache
cdi_path = ARTIFACTS / "cdi_cache_full.pkl"
if not cdi_path.exists():
    cdi_path = ARTIFACTS / "cdi_cache.pkl"
try:
    with open(cdi_path, "rb") as f:
        cdi_cache = pickle.load(f)
    print(f"Loaded CDI cache: {len(cdi_cache)} entries from {cdi_path.name}")
except FileNotFoundError:
    print(f"ERROR: No CDI cache found. Run Phase 3 or scripts/refit_full_gcm_and_cdi.py first.")
    raise


## 3. Build Training Sessions
Construct session objects from the SCM data for the RL environment.

In [ ]:
import ast


def _to_array(val):
    if isinstance(val, (list, np.ndarray)):
        return np.array(val, dtype=np.float32)
    if isinstance(val, str):
        return np.array(ast.literal_eval(val), dtype=np.float32)
    raise TypeError(f"Embedding has unexpected type {type(val)}")


train_sessions = []
for imp_id, group in df.groupby("impression_id", sort=False):
    user_id = group["user_id"].iloc[0]
    history_emb = _to_array(group["U_history_emb_full"].iloc[0])
    item_ids = group["item_id"].tolist()
    clicks = group["Y_click"].tolist()
    title_embs = group["I_title_emb_full"].apply(_to_array).tolist()
    candidates = []
    for iid, emb in zip(item_ids, title_embs):
        candidates.append(type("C", (), {"item_id": iid, "title_emb": emb})())
    session = type("Session", (), {
        "user_id": user_id,
        "initial_history_emb": history_emb,
        "candidates": [candidates],
        "clicks": [clicks],
        "clicked_items": [item_ids[i] for i, c in enumerate(clicks) if c == 1],
    })()
    train_sessions.append(session)

print(f"Built {len(train_sessions)} training sessions from MIND-large")


## 4. Train PPO Agent
Train SB3 PPO with causal diversity reward shaping. More timesteps needed for larger dataset.

In [ ]:
model, save_path = train_ppo(
    train_sessions=train_sessions,
    news_df=df.set_index("item_id"),
    cdi_cache=cdi_cache,
    total_timesteps=1_000_000,
    n_envs=2,
    w=0.6,
    K=20,
    T=1,
    tensorboard_log=str(ARTIFACTS / "tb_logs"),
    checkpoint_dir=str(ARTIFACTS / "checkpoints"),
    verbose=1,
)
print(f"Model saved to: {save_path}")

## 5. Summary

In [ ]:
print(f"Training sessions: {len(train_sessions)}")
print(f"CDI cache entries: {len(cdi_cache)}")
print(f"Checkpoint: {save_path}")
print("\nPhase 4 complete.")